# Gerador de Tiles - D'Aurora

Converte o JP2 em tiles JPEG para o Cloudflare R2.

**Execute as celulas em ordem: 1 → 2 → 3 → 4**

In [ ]:
# CELULA 1 - Instalar dependencias
!pip install imagecodecs Pillow numpy -q
import imagecodecs, numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import os, json, math, time, shutil
print('OK - dependencias instaladas')

In [ ]:
# CELULA 2 - Upload do arquivo JP2 ou JPG
from google.colab import files
print('Selecione o arquivo .jp2 ou .jpg do mapa...')
uploaded = files.upload()
ARQUIVO = list(uploaded.keys())[0]
print(f'Recebido: {ARQUIVO}  ({os.path.getsize(ARQUIVO)/1024/1024:.0f} MB)')

In [ ]:
# CELULA 3 - Gerar tiles
PASTA = 'tiles'
QUALIDADE = 82
TILE_SIZE = 256
FAIXA_PX = 4096

def calcular_niveis(w, h):
    n = 0
    while w > TILE_SIZE or h > TILE_SIZE:
        n += 1; w = (w+1)//2; h = (h+1)//2
    return n

# Abre a imagem
print(f'[1/4] Abrindo {ARQUIVO}...')
t0 = time.time()
ext = ARQUIVO.lower().split('.')[-1]
if ext in ('jp2','j2k','j2c','jpx'):
    print('  Decodificando JP2 (aguarde, pode demorar alguns minutos)...')
    with open(ARQUIVO,'rb') as f: data = f.read()
    arr = imagecodecs.jpeg2k_decode(data)
    del data
    img_full = Image.fromarray(arr).convert('RGB')
    del arr
    USE_FAIXAS = False
else:
    img_ref = Image.open(ARQUIVO)
    w_full, h_full = img_ref.size
    img_ref.close()
    img_full = None
    USE_FAIXAS = True

if not USE_FAIXAS:
    w_full, h_full = img_full.size
print(f'  {w_full}x{h_full} px  ({time.time()-t0:.1f}s)')
max_nivel = calcular_niveis(w_full, h_full)

# Conta tiles
total_tiles = 0
for nivel in range(max_nivel+1):
    scale = 2**nivel
    lw = max(1, math.ceil(w_full/scale))
    lh = max(1, math.ceil(h_full/scale))
    total_tiles += math.ceil(lw/TILE_SIZE) * math.ceil(lh/TILE_SIZE)
print(f'[2/4] Niveis: 0 a {max_nivel}  |  Total de tiles: {total_tiles}')

os.makedirs(PASTA, exist_ok=True)
with open(f'{PASTA}/info.json','w') as f:
    json.dump({'width':w_full,'height':h_full,'tileSize':TILE_SIZE,
               'maxLevel':max_nivel,'format':'jpeg','quality':QUALIDADE}, f, indent=2)

# Gera tiles por faixas
print(f'[3/4] Gerando tiles...')
t0 = time.time()
gerados = 0
faixa_rows = max(1, FAIXA_PX//TILE_SIZE)

for nivel in range(max_nivel+1):
    scale = 2**nivel
    lw = max(1, math.ceil(w_full/scale))
    lh = max(1, math.ceil(h_full/scale))
    cols = math.ceil(lw/TILE_SIZE)
    rows = math.ceil(lh/TILE_SIZE)

    for faixa_inicio in range(0, rows, faixa_rows):
        faixa_fim = min(faixa_inicio+faixa_rows, rows)
        y0_img = faixa_inicio*TILE_SIZE
        y1_img = min(faixa_fim*TILE_SIZE, lh)

        if USE_FAIXAS:
            y0_orig = int(y0_img*scale)
            y1_orig = min(int(y1_img*scale)+1, h_full)
            img_src = Image.open(ARQUIVO)
            if img_src.mode != 'RGB': img_src = img_src.convert('RGB')
            faixa_orig = img_src.crop((0, y0_orig, w_full, y1_orig))
            img_src.close()
            fh = y1_img-y0_img
            faixa = faixa_orig if nivel==0 else faixa_orig.resize((lw, fh), Image.LANCZOS)
            del faixa_orig
        else:
            if nivel==0:
                faixa = img_full.crop((0, y0_img, w_full, y1_img))
            else:
                lv_img = img_full.resize((lw,lh), Image.LANCZOS)
                faixa = lv_img.crop((0, y0_img, lw, y1_img))
                del lv_img

        for row in range(faixa_inicio, faixa_fim):
            row_local = row-faixa_inicio
            ty0 = row_local*TILE_SIZE
            ty1 = min(ty0+TILE_SIZE, faixa.height)
            for col in range(cols):
                x0,x1 = col*TILE_SIZE, min((col+1)*TILE_SIZE, lw)
                tile = faixa.crop((x0, ty0, x1, ty1))
                if tile.size != (TILE_SIZE,TILE_SIZE):
                    pad = Image.new('RGB',(TILE_SIZE,TILE_SIZE),(0,0,0))
                    pad.paste(tile,(0,0)); tile=pad
                col_dir = f'{PASTA}/{nivel}/{col}'
                os.makedirs(col_dir, exist_ok=True)
                tile.save(f'{col_dir}/{row}.jpg','JPEG',quality=QUALIDADE,optimize=True)
                gerados += 1
        del faixa

        elapsed = time.time()-t0
        vel = gerados/elapsed if elapsed>0 else 1
        eta = int((total_tiles-gerados)/vel)
        print(f'  Nivel {nivel}/{max_nivel}  {gerados}/{total_tiles} ({gerados/total_tiles*100:.0f}%)  ETA: {eta//60}m{eta%60:02d}s        ', end='\r')

elapsed = time.time()-t0
tamanho = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(PASTA) for f in fs)
print(f'\n[4/4] Concluido em {int(elapsed//60)}m{int(elapsed%60):02d}s')
print(f'      {tamanho/1024/1024:.1f} MB  |  {total_tiles} tiles')

In [ ]:
# CELULA 4 - Baixar tiles como ZIP
from google.colab import files
print('Compactando...')
shutil.make_archive('tiles', 'zip', '.', 'tiles')
print(f'ZIP: {os.path.getsize("tiles.zip")/1024/1024:.1f} MB')
files.download('tiles.zip')

## Apos o download

Extraia o ZIP e suba no R2:
```bash
wrangler r2 object put-batch SEU-BUCKET/tiles/ --dir tiles/
```

No admin, aba Mapa Historico, troque a URL por:
```
https://pub-xxxxxxxx.r2.dev/tiles/info.json
```